In [8]:
!pip install transformers torch

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [10]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [11]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day 😊")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Limit chat history (IMPORTANT FIX)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids[:, -500:], new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate better response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        min_length=20,
        do_sample=True,
        top_k=100,
        top_p=0.92,
        temperature=0.7,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: HELLO
Chatbot: Hey, hey! How's it going? D: Uhm... what's wrong?
You: WHAT IS AI
Chatbot: We are all an AI.
You: EXIT
Chatbot: Goodbye! Have a great day 😊
